In [2]:
!pip install langchain langchain-community langchain-core
!pip install langchain-huggingface
!pip install langchain-groq
!pip install faiss-cpu
!pip install sentence-transformers
!pip install python-dotenv
!pip install langchain_ollama
!pip install langchain_unstructured
!pip install langchain_community
!pip install langchain_text_splitters
!pip install langchain_experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.15
    Uninstalling langchain-core-1.2.15:
      Successfully uninstalled langchain-core-1.2.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>

In [3]:
from langchain_ollama.llms import OllamaLLM
from langchain_unstructured import UnstructuredLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.documents import Document # Added import for Document
from langchain_groq import ChatGroq
import os
# import streamlit as st
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

In [4]:
load_dotenv()
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [5]:
os.environ["GROQ_API_KEY"] ="YOUR_API_KEY"

In [6]:
import os
import sys
# from pathlib import Path
# from llama_cpp import Llama

In [99]:
llm = ChatGroq(
  model="llama-3.1-8b-instant",
  temperature=0.1,
  max_retries=2
)

In [100]:
print(llm.invoke("Hello!"))

content="Hello! It's nice to meet you. Is there something I can help you with or would you like to chat?" additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 37, 'total_tokens': 62, 'completion_time': 0.025235802, 'completion_tokens_details': None, 'prompt_time': 0.003000195, 'prompt_tokens_details': None, 'queue_time': 0.005931246, 'total_time': 0.028235997}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019db7fd-ad91-74d0-ab49-cb52ad85689a-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 37, 'output_tokens': 25, 'total_tokens': 62}


In [9]:
file_paths = [
    "/kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv",
    "/kaggle/input/competitions/llm-agentic-legal-information-retrieval/laws_de.csv"
]
#court_considerations.csv

In [10]:
import pandas as pd
from langchain_core.documents import Document # Added for self-sufficiency

In [11]:
#Already created
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# embedding_model = HuggingFaceEmbeddings(
#     model_name="intfloat/multilingual-e5-base",
#     model_kwargs={
#         "device": "cuda"
#     },
#     encode_kwargs={
#         "normalize_embeddings": True,
#         "batch_size": 64
#     }
# )

# from sentence_transformers import SentenceTransformer
import kagglehub

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en",
    model_kwargs={
        "device": "cuda"
    },
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 64
    }
)

from tqdm.auto import tqdm

def build_faiss_streaming(file_paths, save_path="/kaggle/working/faiss_index"):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100
    )

    vectorstore = None

    # File-level progress
    for file_path in tqdm(file_paths, desc="Processing files"):

        if not file_path.endswith(".csv"):
            continue

        try:
            # Chunk-level progress
            chunk_iter = pd.read_csv(
                file_path,
                chunksize=800,
                on_bad_lines="skip",
                engine="python"
            )

            for chunk_df in tqdm(chunk_iter, desc=f"Reading {file_path}", leave=False):

                docs_batch = []

                for row in chunk_df.itertuples(index=False):
                    metadata = row._asdict()

                    if hasattr(row, "text") and hasattr(row, "citation"):
                        docs_batch.append(
                            Document(
                                page_content=str(row.text),
                                metadata={"citation": str(row.citation)}
                            )
                        )
                    else:
                        page_content = " ".join(
                            [str(v) for v in metadata.values() if pd.notna(v)]
                        )
                        docs_batch.append(
                            Document(
                                page_content=page_content,
                                metadata=metadata
                            )
                        )

                # Splitting progress
                split_docs = text_splitter.split_documents(docs_batch)

                #  Embedding + FAISS progress
                if vectorstore is None:
                    print("Building FAISS index...")
                    vectorstore = FAISS.from_documents(split_docs, embedding_model)
                else:
                    vectorstore.add_documents(split_docs)

        except Exception as e:
            print(f" Error processing {file_path}: {e}")

    if vectorstore:
        print(" Saving FAISS index...")
        vectorstore.save_local(save_path)

    print(" Done!")
    return vectorstore

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
from langchain_core.load import dumps, loads

In [14]:
import os

INDEX_PATH = "/kaggle/input/datasets/shikhachandelcollege/faiss-index"
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en",
    model_kwargs={
        "device": "cuda"
    },
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 64
    }
)

if os.path.exists(INDEX_PATH):
    print("Loading existing FAISS index...")
    real_vector_store = FAISS.load_local(
        INDEX_PATH,
        embedding_model,
        allow_dangerous_deserialization=True
    )
else:
    print("Building FAISS index...")
    real_vector_store = build_faiss_streaming(file_paths, INDEX_PATH)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading existing FAISS index...


In [110]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
# reranker = CrossEncoder("BAAI/bge-reranker-large")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [102]:
def balance_citations(ranked):
    art = [c for c in ranked if c.startswith("Art.")]
    bge = [c for c in ranked if "BGE" in c]
    others = [c for c in ranked if c not in art and c not in bge]

    # prioritize statutes first
    return art[:30] + bge[:15] + others[:5]
    
def rerank_citations(query, citation_map):
    citation_texts = []
    citation_list = []

    query_l = query.lower()

    for citation, texts in citation_map.items():
        merged_text = " ".join(texts[:3])
        citation_texts.append((query, merged_text))
        citation_list.append(citation)

    scores = reranker.predict(citation_texts)

    adjusted_scores = []

    for citation, score, (_, text_pair) in zip(citation_list, scores, citation_texts):
        text = text_pair.lower()

        # German legal terms boost
        if any(word in text for word in ["gesetz", "artikel"]):
            score += 0.05

        # English/French legal terms boost
        if any(word in text for word in ["article", "loi"]):
            score += 0.05

        adjusted_scores.append((citation, score))

    ranked = sorted(adjusted_scores, key=lambda x: x[1], reverse=True)

    return [c for c, _ in ranked]

In [111]:
def format_query(q):
    return " Represent this legal question for retrieving relevant Swiss law citations across languages: " +q

In [112]:
def retrieve_docs(query):
    formatted_query = format_query(query)
    return real_vector_store.similarity_search(formatted_query, k=1200)

In [120]:
template = """You are a Swiss legal citation expert. Output ONLY a Python list of citations.

CITATION FORMATS:
- Federal laws: "Art. X ABBREV"
- Court decisions: "BGE X Y Z" or "BGE X Y Z E. N"

Given the query and candidate citations, select the most relevant ones.

Query:
{query}

Candidate Citations:
{candidates}

Return ONLY a Python list.

Answer:

"""
prompt_perspectives = PromptTemplate.from_template(template)

from langchain_core.output_parsers import StrOutputParser



citation_selector = (
    prompt_perspectives 
    | llm
    | StrOutputParser()
)

In [123]:
from collections import defaultdict

def generate_queries_multilingual(question):
    base_queries = [
        question,
        f"Swiss law article about {question}",
        f"relevant legal provisions {question}",
        f"case law related to {question}"
    ]

    return base_queries

def clean_citations(citations):
    pattern = re.compile(r"(Art\. \d+.*|BGE \d+ .*|[0-9A-Z]+_\d+/\d{4}.*)")
    return [c for c in citations if pattern.match(c)]

def retrieve_agentic(question, k=200, top_n=50):

    queries = generate_queries_multilingual(question)

    all_docs = []

    # Step 1: Retrieve across all language variants
    for q in queries:
        formatted_q = format_query(q)
        docs = real_vector_store.similarity_search(formatted_q, k=k)
        all_docs.extend(docs)

    # Step 2: Deduplicate
    all_docs = deduplicate_docs(all_docs)

    # Step 4: Group citations (with extraction)
    citation_map = group_by_citation(all_docs)

    # Step 5: Global rerank
    expanded_query = " ".join(queries)
    ranked_citations = rerank_citations(expanded_query, citation_map)

    # Step 6: Balance output

    candidate_citations = balance_citations(ranked_citations)[:top_n]

    #Step 6: use llm
    
    llm_output = citation_selector.invoke({
        "query": question,
        "candidates": candidate_citations
        # "citation_texts": citation_texts
    })

    try:
        final_citations = eval(llm_output)
    except:
        final_citations = candidate_citations[:top_n]

    return final_citations



def deduplicate_docs(docs):
    seen = set()
    unique_docs = []

    for doc in docs:
        key = (doc.page_content, doc.metadata.get("citation", ""))

        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)

    return unique_docs
    
def extract_citations(docs):
    seen = set()
    citations = []

    for doc in docs:
        citation = doc.metadata.get("citation")
        if citation and citation not in seen:
            seen.add(citation)
            citations.append(citation)

    return citations


def group_by_citation(docs):
    citation_map = defaultdict(list)

    for doc in docs:
        text = doc.page_content

        # 1. Extract from metadata (if exists)
        meta_citation = doc.metadata.get("citation")

        if meta_citation:
            citation_map[meta_citation].append(text)

        # 2. Extract from text (NEW)
        extracted = extract_legal_citations(text)

        for citation in extracted:
            citation_map[citation].append(text)

    return citation_map

In [124]:
import re

LAW_ABBREVS = (
    "ZGB|OR|StGB|BV|SchKG|ZPO|StPO|BGG|VwVG|IPRG|KG|DSG|MSchG|URG|PatG|"
    "DesG|UWG|PrSG|FINMAG|BankG|VAG|KAG|GwG|BEHG|FinfraG|FIDLEG|FINIG|"
    "ATSG|AHV|IV|EO|ALV|KVG|UVG|BVG|ArG|GlG|USG|RPG|WaG|JSG|TSchG|"
    "LwG|PBG|EBG|SVG|LFG|SebG|SpG|BoeB|EMRK|SR|AS|BBl|ParlG|RVOG|RVOV|"
    "MG|BPG|BPV|VBGÖ|VDSG|MWSTG|DBG|StHG|VStG|StG|ZG|CO|CP|CC|CPC|CPP|"
    "LEtr|LAsi|LN|LDIP|LCart|LDA|LPM|LBI|LDes|LCD|LFINMA|LB|LSA|LPCC|"
    "LBA|LBVM|LIMF|LSFin|LEFin|LAVS|LAI|LAPG|LACI|LAMal|LAA|LPP|LTr|"
    "LEg|LPD|LPE|LAT|LFo|LChP|LPN|LAgr|LTV|LCdF|LNA|LPTh|LTAF|LTF"
)

ART_PATTERN = re.compile(
    rf"""
    Art\.\s*
    (\d+[a-zA-Z]*)                 # Article number (e.g., 28, 28a)
    (?:\s*Abs\.\s*\d+[a-zA-Z]*)?   # Optional Abs.
    (?:\s*lit\.\s*[a-z])?          # Optional lit.
    \s+
    ({LAW_ABBREVS})               # Law abbreviation
    """,
    re.VERBOSE
)

BGE_PATTERN = re.compile(
    r"BGE\s+\d+\s+[IVXLC]+\s+\d+(?:\s+E\.\s*[\d\.a-z]+)?"
)

CASE_PATTERN = re.compile(
    r"\b\d+[A-Z]_\d+/\d{4}(?:\s+E\.\s*[\d\.a-z]+)?"
)

In [125]:
def normalize_citation(c):
    c = c.strip()

    # Normalize Art.
    c = re.sub(r"\bART\b", "Art.", c, flags=re.IGNORECASE)
    c = re.sub(r"\bArt\s+", "Art. ", c)

    # Remove duplicate spaces
    c = re.sub(r"\s+", " ", c)

    return c
    
def is_valid_citation(c):
    return (
        ART_PATTERN.match(c)
        or BGE_PATTERN.match(c)
        or CASE_PATTERN.match(c)
    )
    
def extract_legal_citations(text):
    citations = set()

    # Art. citations
    for match in ART_PATTERN.finditer(text):
        full = match.group(0)
        citations.add(full.strip())

    # BGE citations
    for match in BGE_PATTERN.finditer(text):
        citations.add(match.group(0).strip())

    # Case law citations
    for match in CASE_PATTERN.finditer(text):
        citations.add(match.group(0).strip())

    return list(citations)

In [126]:
import pandas as pd
import numpy as np
# Load validation dataset
queries_df = pd.read_csv("/kaggle/input/competitions/llm-agentic-legal-information-retrieval/val.csv")

results = []

for _, row in queries_df.iterrows():

    query_id = row["query_id"]
    question = row["query"]
    gold = row["gold_citations"]

    try:
        citations = retrieve_agentic(question, k=200, top_n=100)
        predicted = ";".join(citations)

    except Exception as e:
        print(f"Error: {e}")
        predicted = ""

    results.append({
        "query_id": query_id,
        "gold_citations": gold,
        "predicted_citations": predicted
    })

# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Save predictions
results_df.to_csv("/kaggle/working/val_submission.csv", index=False)
print("Predictions saved.")
print(results)

Predictions saved.
[{'query_id': 'val_001', 'gold_citations': 'Art. 221 Abs. 1 StPO;Art. 140 Abs. 1 StGB;Art. 396 Abs. 1 StPO;Art. 222 StPO;Art. 393 Abs. 1 StPO;Art. 382 Abs. 1 StPO;Art. 385 Abs. 1 StPO;Art. 221 Abs. 2 StPO;Art. 227 Abs. 1 StPO;Art. 212 Abs. 3 StPO;Art. 390 Abs. 2 StPO;Art. 422 Abs. 1 StPO;Art. 422 Abs. 2 StPO;Art. 428 Abs. 1 StPO;Art. 135 Abs. 4 StPO;Art. 100 Abs. 1 BGG;Art. 135 Abs. 3 StPO;Art. 37 Abs. 1 StBOG;Art. 39 Abs. 1 StBOG;BGE 137 IV 122 E. 6.2;BGE 137 IV 122 E. 6.4;BGE 137 IV 122 E. 4.2;BGE 132 I 21 E. 3.2;1B_210/2023 E. 4.1;BGE 132 I 21 E. 3.2.2;1B_536/2018 E. 5.1;BGE 139 IV 270 E. 3.1;BGE 133 I 168 E. 4.1;BGE 143 IV 168 E. 5.1;BGE 133 I 270 E. 3.4.2;BGE 137 IV 122 E. 4.1;BGE 132 I 21 E. 3.2.1;1B_90/2021 E. 2.1;1B_90/2021 E. 2.4;7B_496/2025 E. 3.2;7B_231/2025 E. 4.1;7B_69/2024 E. 3.3.2;7B_301/2024 E. 2.4;7B_12/2025 E. 2.2;1B_357/2022 E. 3.1;1B_15/2023 E. 3.1;1B_28/2022 E. 4.1', 'predicted_citations': 'BGE 139 IV 191 E. 4.1;BGE 139 IV 191;BGE 137 IV 186 E. 3

In [127]:
from sklearn.metrics import f1_score
import pandas as pd

def clean_citations(citations):
    if pd.isna(citations):
        return set()

    items = str(citations).split(";")

    cleaned = set()
    for c in items:
        c = c.strip()
        if not c:
            continue

        # normalize (reuse your function)
        c = normalize_citation(c)

        cleaned.add(c)

    return cleaned

def compute_metrics(row):

    gold = clean_citations(row["gold_citations"])
    pred = clean_citations(row["predicted_citations"])

    tp = len(gold & pred)

    precision = tp / len(pred) if len(pred) > 0 else 0
    recall = tp / len(gold) if len(gold) > 0 else 0

    f1 = 0 if (precision + recall == 0) else 2 * precision * recall / (precision + recall)

    return pd.Series([precision, recall, f1])


results_df[["precision","recall","f1"]] = results_df.apply(compute_metrics, axis=1)

print("Average Precision:", results_df["precision"].mean())
print("Average Recall:", results_df["recall"].mean())
print("Average F1:", results_df["f1"].mean())     

Average Precision: 0.06666666666666667
Average Recall: 0.01111111111111111
Average F1: 0.01904761904761905


In [ ]:
queries_df = pd.read_csv("/kaggle/input/competitions/llm-agentic-legal-information-retrieval/test.csv")

results = []

for _, row in queries_df.iterrows():

    query_id = row["query_id"]
    question = row["query"]
    # gold = row["gold_citations"]

    try:
        citations = retrieve_agentic(question, k=200, top_n=100)
        predicted = ";".join(citations)

    except Exception as e:
        print(f"Error: {e}")
        predicted = ""

    results.append({
        "query_id": query_id,
        # "gold_citations": gold,
        "predicted_citations": predicted
    })


# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Save predictions
results_df.to_csv("/kaggle/working/test_submission.csv", index=False)

print("Predictions saved.")
print(results)
